## To mount the Google Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Extract Zip file

In [6]:
import zipfile

path_to_zip_file = '/content/drive/MyDrive/Project/Preprocessed_Data.zip'
directory_to_extract_to = '/content/drive/MyDrive/Project/'

with zipfile.ZipFile(path_to_zip_file, 'r') as zip_ref:
    zip_ref.extractall(directory_to_extract_to)


## Install Libraries

In [1]:
#!pip install torch torchvision torchaudio timm scikit-learn numpy matplotlib pillow

In [3]:
import torch
print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())

2.10.0+cu128
CUDA available: True


## Choose Directory

In [5]:
cd /content/drive/MyDrive/Project

/content/drive/MyDrive/Project


## Analyse with Deep CNN models

In [ ]:
import torch
import torch.nn as nn
import json
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ==============================
# CONFIG
# ==============================
DATASET_PATH = "Preprocessed_Data"
BATCH_SIZE = 16
EPOCHS = 20
LR = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAMES = [
    "alexnet",
    "vgg16",
    "resnet50",
    "googlenet",
    "densenet121",
    "mobilenet_v2",
    "efficientnet_b0"
]

# ==============================
# TRANSFORMS
# ==============================
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# ==============================
# LOAD DATASET
# ==============================
full_dataset = datasets.ImageFolder(DATASET_PATH)
targets = full_dataset.targets

train_idx, val_idx = train_test_split(
    np.arange(len(targets)),
    test_size=0.3,
    stratify=targets,
    random_state=42
)

train_data = Subset(
    datasets.ImageFolder(DATASET_PATH, transform=train_transform),
    train_idx
)

val_data = Subset(
    datasets.ImageFolder(DATASET_PATH, transform=val_transform),
    val_idx
)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=BATCH_SIZE)

NUM_CLASSES = len(full_dataset.classes)

# ==============================
# MODEL LOADER FUNCTION
# ==============================
def get_model(name):
    if name == "alexnet":
        model = models.alexnet(pretrained=True)
        model.classifier[6] = nn.Linear(4096, NUM_CLASSES)

    elif name == "vgg16":
        model = models.vgg16(pretrained=True)
        model.classifier[6] = nn.Linear(4096, NUM_CLASSES)

    elif name == "resnet50":
        model = models.resnet50(pretrained=True)
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

    elif name == "googlenet":
        model = models.googlenet(pretrained=True)
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

    elif name == "densenet121":
        model = models.densenet121(pretrained=True)
        model.classifier = nn.Linear(model.classifier.in_features, NUM_CLASSES)

    elif name == "mobilenet_v2":
        model = models.mobilenet_v2(pretrained=True)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)

    elif name == "efficientnet_b0":
        model = models.efficientnet_b0(pretrained=True)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)

    else:
        raise ValueError("Unknown model")

    return model

# ==============================
# TRAIN FUNCTION
# ==============================
def train_model(model, name):
    print(f"\nTraining: {name}")
    model.to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(images)

            if isinstance(outputs, tuple):
                outputs = outputs[0]

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"{name} Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss:.4f}")

    return model

# ==============================
# EVALUATE FUNCTION
# ==============================
def evaluate_model(model, name):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)

            outputs = model(images)
            if isinstance(outputs, tuple):
                outputs = outputs[0]

            preds = torch.argmax(outputs, dim=1).cpu()

            y_true.extend(labels.numpy())
            y_pred.extend(preds.numpy())

    acc = accuracy_score(y_true, y_pred)

    # Print classification report
    print(f"\n{name} Classification Report:")
    print(classification_report(y_true, y_pred, target_names=full_dataset.classes))

    report = classification_report(
        y_true, y_pred,
        target_names=full_dataset.classes,
        output_dict=True
    )

    cm = confusion_matrix(y_true, y_pred)

    return acc, report, cm

# ==============================
# SAVE CONFUSION MATRIX
# ==============================
def save_confusion_matrix(cm, name):
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d",
                xticklabels=full_dataset.classes,
                yticklabels=full_dataset.classes)

    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"{name} Confusion Matrix")

    plt.savefig(f"{name}_confusion_matrix.png")
    plt.close()

# ==============================
# SAVE MODEL
# ==============================
def save_model(model, name):
    torch.save({
        "model_name": name,
        "model_state_dict": model.state_dict(),
        "num_classes": NUM_CLASSES,
        "class_names": full_dataset.classes
    }, f"{name}_model.pth")

# ==============================
# LOAD MODEL
# ==============================
def load_model(filepath):
    checkpoint = torch.load(filepath, map_location=DEVICE)

    model_name = checkpoint["model_name"]
    class_names = checkpoint["class_names"]

    model = get_model(model_name)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(DEVICE)
    model.eval()

    return model, class_names

# ==============================
# MAIN TRAINING LOOP
# ==============================
results = {}

for name in MODEL_NAMES:
    model = get_model(name)

    model = train_model(model, name)

    acc, report, cm = evaluate_model(model, name)

    print(f"{name} Accuracy: {acc:.4f}")

    # Save everything
    save_model(model, name)
    save_confusion_matrix(cm, name)

    with open(f"{name}_classification_report.json", "w") as f:
        json.dump(report, f, indent=4)

    results[name] = {
        "accuracy": float(acc)
    }

# ==============================
# SAVE FINAL RESULTS
# ==============================
with open("cnn_results.json", "w") as f:
    json.dump(results, f, indent=4)

with open("class_names.json", "w") as f:
    json.dump(full_dataset.classes, f, indent=4)

print("\nFinal Results:")
print(results)

# ==============================
# TEST LOADED MODEL
# ==============================
print("\nTesting saved model...")

model, class_names = load_model(f"{MODEL_NAMES[0]}_model.pth")

image, _ = val_data[0]
image = image.unsqueeze(0).to(DEVICE)

with torch.no_grad():
    output = model(image)
    pred = torch.argmax(output, dim=1).item()

print("Predicted class:", class_names[pred])

## Vit Based models

In [ ]:
import torch
import torch.nn as nn
import timm
import json
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ==============================
# CONFIG
# ==============================
DATASET_PATH = "Preprocessed_Data"
BATCH_SIZE = 16
EPOCHS = 20
LR = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAMES = [
    "vit_base_patch16_224",
    "vit_small_patch16_224",
    "deit_base_patch16_224",
    "swin_tiny_patch4_window7_224"
]

# ==============================
# TRANSFORMS
# ==============================
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# ==============================
# LOAD DATASET
# ==============================
full_dataset = datasets.ImageFolder(DATASET_PATH)
targets = full_dataset.targets

train_idx, val_idx = train_test_split(
    np.arange(len(targets)),
    test_size=0.3,
    stratify=targets,
    random_state=42
)

train_data = Subset(
    datasets.ImageFolder(DATASET_PATH, transform=train_transform),
    train_idx
)

val_data = Subset(
    datasets.ImageFolder(DATASET_PATH, transform=val_transform),
    val_idx
)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=BATCH_SIZE)

NUM_CLASSES = len(full_dataset.classes)

# ==============================
# CREATE MODEL FUNCTION
# ==============================
def get_model(model_name):
    model = timm.create_model(model_name, pretrained=True)

    if hasattr(model, 'head'):
        model.head = nn.Linear(model.head.in_features, NUM_CLASSES)
    elif hasattr(model, 'fc'):
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

    return model

# ==============================
# TRAIN FUNCTION
# ==============================
def train_model(model, name):
    print(f"\nTraining: {name}")
    model.to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"{name} Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss:.4f}")

    return model

# ==============================
# EVALUATE FUNCTION
# ==============================
def evaluate_model(model, name):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1).cpu()

            y_true.extend(labels.numpy())
            y_pred.extend(preds.numpy())

    acc = accuracy_score(y_true, y_pred)

    print(f"\n{name} Classification Report:")
    print(classification_report(y_true, y_pred, target_names=full_dataset.classes))

    report = classification_report(
        y_true, y_pred,
        target_names=full_dataset.classes,
        output_dict=True
    )

    cm = confusion_matrix(y_true, y_pred)

    return acc, report, cm

# ==============================
# SAVE CONFUSION MATRIX
# ==============================
def save_confusion_matrix(cm, name):
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d",
                xticklabels=full_dataset.classes,
                yticklabels=full_dataset.classes)

    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"{name} Confusion Matrix")

    plt.savefig(f"{name}_confusion_matrix.png")
    plt.close()

# ==============================
# SAVE MODEL WITH METADATA
# ==============================
def save_model(model, name):
    torch.save({
        "model_name": name,
        "model_state_dict": model.state_dict(),
        "num_classes": NUM_CLASSES,
        "class_names": full_dataset.classes
    }, f"{name}_model.pth")

# ==============================
# LOAD MODEL
# ==============================
def load_model(filepath):
    checkpoint = torch.load(filepath, map_location=DEVICE)

    model_name = checkpoint["model_name"]
    class_names = checkpoint["class_names"]

    model = get_model(model_name)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(DEVICE)
    model.eval()

    return model, class_names

# ==============================
# MAIN LOOP
# ==============================
results = {}

for model_name in MODEL_NAMES:
    model = get_model(model_name)

    model = train_model(model, model_name)

    acc, report, cm = evaluate_model(model, model_name)

    print(f"{model_name} Accuracy: {acc:.4f}")

    # Save everything
    save_model(model, model_name)
    save_confusion_matrix(cm, model_name)

    with open(f"{model_name}_classification_report.json", "w") as f:
        json.dump(report, f, indent=4)

    results[model_name] = {
        "accuracy": float(acc)
    }

# ==============================
# SAVE FINAL RESULTS
# ==============================
with open("vit_results.json", "w") as f:
    json.dump(results, f, indent=4)

with open("class_names.json", "w") as f:
    json.dump(full_dataset.classes, f, indent=4)

print("\nFinal Results:")
print(results)

# ==============================
# TEST LOADED MODEL
# ==============================
print("\nTesting saved model...")

model, class_names = load_model(f"{MODEL_NAMES[0]}_model.pth")

image, _ = val_data[0]
image = image.unsqueeze(0).to(DEVICE)

with torch.no_grad():
    output = model(image)
    pred = torch.argmax(output, dim=1).item()

print("Predicted class:", class_names[pred])